In [ ]:
#list of available datasets with available control and perturbed data
#The following datasets make use of crispr screening such as ShRNA, CRISPRi, CRISPRa and CRISPRKO
#https://virtualcellchallenge.org/datasets and https://colab.research.google.com/drive/1QKOtYP7bMpdgDJEipDxaJqOchv7oQ-_l#scrollTo=wShYx6z-ajtN
#https://zenodo.org/records/15731884

#Cytokine Perturbations
#https://figshare.com/articles/dataset/pbmc_parse/28589774?file=53372768
#https://support.parsebiosciences.com/hc/en-us/articles/7704577188500-How-to-analyze-a-1-million-cell-data-set-using-Scanpy-and-Harmony

In [ ]:
#The following steps should be adjusted based on the dataset under consideration
# The insilico perturbation reinforcement learning environment can be modified to target specific gene(s) for the
#following datasets
#https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE134139
#https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE119352

In [ ]:
from zipfile import ZipFile
from tqdm.auto import tqdm
import os
%pip install requests tqdm
#This cell of code was adapted from the virtual cell challenge
#https://colab.research.google.com/drive/1QKOtYP7bMpdgDJEipDxaJqOchv7oQ-_l#scrollTo=wShYx6z-ajtN
import requests
from tqdm.auto import tqdm  # picks the best bar for the environment

website = "https://storage.googleapis.com/vcc_data_prod/datasets/state/competition_support_set.zip"
output_path = "competition_support_set.zip"


response = requests.get(website, stream=True)
total = int(response.headers.get("content-length", 0))

with open(output_path, "wb") as f, tqdm(
    total=total, unit='B', unit_scale=True, desc="Downloading"
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        if not chunk:
            break
        f.write(chunk)
        bar.update(len(chunk))
out_dir  = "competition_support_set"

os.makedirs(out_dir, exist_ok=True)

with ZipFile(output_path, 'r') as z:
    for member in tqdm(z.infolist(), desc="Unzipping", unit="file"):
        z.extract(member, out_dir)

Downloading:   0%|          | 0.00/8.72G [00:00<?, ?B/s]

Unzipping:   0%|          | 0/12 [00:00<?, ?file/s]

In [ ]:
!7z x 1M_PBMC_T1D_Parse.zip -o1M_PBMC_T1D_Parse


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan         1 file, 3618245992 bytes (3451 MiB)

Extracting archive: 1M_PBMC_T1D_Parse.zip
--
Path = 1M_PBMC_T1D_Parse.zip
Type = zip
Physical Size = 3618245992

  0%      0% 1 - cell_metadata_1M_PBMC.csv                                    0% 2        0% 2 - DGE_1M_PBMC.mtx                          1% 2 - DGE_1M_PBMC.mtx                          2% 2 - DGE_1M_PBMC.mtx                          3% 2 - DGE_1M_PBMC.mtx                          4% 2 - DGE_1M_PBMC.mt

In [ ]:
import scanpy as sc
import numpy as np
from scipy.sparse import csr_matrix

adata = sc.read_mtx("1M_PBMC_T1D_Parse/DGE_1M_PBMC.mtx").T

sc.pp.subsample(adata, n_obs=20_000, random_state=42)

sc.pp.filter_genes(adata, min_cells=10)

adata.X = adata.X.astype(np.float32)
adata.X = csr_matrix(adata.X)

adata.write(
    "PBMC_Parse_20k.h5ad",
    compression="gzip",
    compression_opts=9
)



In [ ]:
import scanpy as sc
import numpy as np

# ----------------------------
# CONFIG
# ----------------------------
INPUT_H5AD  = "competition_support_set/competition_support_set/competition_train.h5"
OUTPUT_H5AD = "train_20k.h5ad"
N_TARGET    = 20_000 # the target can be increase by  considering availability of RAM
SEED        = 42

np.random.seed(SEED)

# ----------------------------
# LOAD (disk-backed)
# ----------------------------
adata = sc.read_h5ad(INPUT_H5AD, backed="r")
print("Loaded (backed):", adata)

# ----------------------------
# RANDOM SAMPLE
# ----------------------------
n_obs = adata.n_obs

if N_TARGET >= n_obs:
    indices = np.arange(n_obs)
else:
    indices = np.random.choice(n_obs, size=N_TARGET, replace=False)

print("Selected cells:", len(indices))

# ----------------------------
# LOAD REDUCE DATA  INTO MEMORY
# ----------------------------
adata_reduce_small = adata[indices].to_memory()

print("Final dataset:", adata_reduce_small)
print("Obs columns:", list(adata_reduce_small.obs.columns))
print("Var columns:", list(adata_reduce_small.var.columns))

# ----------------------------
# SAVE
# ----------------------------
adata_reduce_small.write_h5ad(OUTPUT_H5AD)
print(f"Saved → {OUTPUT_H5AD}")


Loaded (backed): AnnData object with n_obs × n_vars = 221273 × 18080 backed at 'competition_support_set/competition_support_set/competition_train.h5'
    obs: 'target_gene', 'guide_id', 'batch', 'batch_var', 'cell_type'
    uns: 'log1p'
Selected cells: 20000
Final dataset: AnnData object with n_obs × n_vars = 20000 × 18080
    obs: 'target_gene', 'guide_id', 'batch', 'batch_var', 'cell_type'
    uns: 'log1p'
Obs columns: ['target_gene', 'guide_id', 'batch', 'batch_var', 'cell_type']
Var columns: []
Saved → train_20k.h5ad


In [ ]:
#

# modify based on the selected dataset from the first cell

import scanpy as sc
# Virtual Cell Challenge Dataset
adata = sc.read_h5ad("train_20k.h5ad")
adata = adata.to_memory()

# Virtual Cell Challenge Dataset
control_mask = adata.obs["target_gene"] == "non-targeting"
perturb_mask = adata.obs["target_gene"] != "non-targeting"


# cytokine Dataset
#adata = sc.read_h5ad("cytokine_20k.h5ad")
#control_mask = adata.obs["treatment"] == "PBS"
#perturb_mask = adata.obs["treatment"] != "PBS"


adata_ctrl = adata[control_mask].copy()
adata_pert = adata[perturb_mask].copy()
#adata_ctrl
#adata_pert

# Clonal Aware Data(https://zenodo.org/records/15731884)
#adata_ctrl = sc.read_10x_h5("filtered_feature_bc_matrix_control1.h5")
#adata_pert = sc.read_10x_h5("filtered_feature_bc_matrix_sdma.h5")

adata_ctrl.var_names_make_unique()
adata_pert.var_names_make_unique()




import os
import numpy as np
import scipy.io as sio
from scipy import sparse

def export_to_mtx(adata, outdir):
    os.makedirs(outdir, exist_ok=True)

    # ---- Expression matrix (cells × genes → genes × cells for MTX) ----
    X = adata.X
    if not sparse.issparse(X):
        X = sparse.csr_matrix(X)

    sio.mmwrite(
        os.path.join(outdir, "matrix.mtx"),
        X.T  # MTX expects genes × cells
    )

    # ---- Gene names ----
    genes = adata.var_names.to_numpy()
    np.savetxt(
        os.path.join(outdir, "genes.tsv"),
        genes,
        fmt="%s"
    )

    # ---- Cell barcodes (recommended) ----
    barcodes = adata.obs_names.to_numpy()
    np.savetxt(
        os.path.join(outdir, "barcodes.tsv"),
        barcodes,
        fmt="%s"
    )

    print(f"Exported to {outdir}")
    print("  matrix.mtx (genes × cells)")
    print("  genes.tsv")
    print("  barcodes.tsv")

export_to_mtx(adata_ctrl, "Control_train Data")
export_to_mtx(adata_pert, "Perturbed_train Data")



Exported to Control_train Data
  matrix.mtx (genes × cells)
  genes.tsv
  barcodes.tsv
Exported to Perturbed_train Data
  matrix.mtx (genes × cells)
  genes.tsv
  barcodes.tsv


In [ ]:
import torch
from torch.optim import Optimizer

import torch
from torch.optim import Optimizer

class ASGDAMSGrad(Optimizer):
    """Proposed ASGD (Improved Adam + AMSGrad) with decoupled weight decay."""

    def __init__(self, params, lr=None, beta1=0.9, beta2=0.9, eps=1e-8,
                 lr_min=1e-4, lr_max=3e-4, weight_decay=0):
        defaults = dict(beta1=beta1, beta2=beta2, eps=eps,
                        lr_min=lr_min, lr_max=lr_max,
                        weight_decay=weight_decay)
        super().__init__(params, defaults)
        self.f_min = 0
        self.f_max = 0
        self.last_lr = lr_max

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        total_nonzero_fmin, total_nonzero_fmax = 0, 0

        for group in self.param_groups:
            beta1, beta2, eps = group['beta1'], group['beta2'], group['eps']
            wd = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                # Decoupled weight decay
                if wd != 0:
                    p.mul_(1 - wd)

                g = p.grad
                state = self.state[p]

                if not state:
                    state['t'] = 0
                    state['n'] = torch.zeros_like(p)
                    state['H'] = torch.zeros_like(p)
                    state['H_hat'] = torch.zeros_like(p)

                n, H, H_hat = state['n'], state['H'], state['H_hat']
                state['t'] += 1

                n.mul_(beta1).add_(g, alpha=1 - beta1)
                H.mul_(beta2).addcmul_(g, g, value=1 - beta2)
                torch.maximum(H_hat, H, out=H_hat)

                mean_abs_H = n.abs().mean()
                mean_H = n.mean()

                f_min = (mean_abs_H).sign() - (mean_H).sign()
                f_max = (mean_H).sign() - (mean_abs_H).sign()
              #  f_min = (mean_abs_H - mean_H)
              #  f_max = (mean_H - mean_abs_H)

                #total_nonzero_fmin += (f_min != 0).sum().item()
                #total_nonzero_fmax += (f_max != 0).sum().item()

                denom = H_hat.sqrt().add(eps)
                state['step_dir'] = n / denom

       # use_lr_min = (total_nonzero_fmax <   total_nonzero_fmin)
        use_lr_min = f_max ==   f_min
        self.f_min = f_min
        self.f_max = f_max

        for group in self.param_groups:
            lr = group['lr_min'] if use_lr_min else group['lr_max']
            self.last_lr = lr

            for p in group['params']:
                if p.grad is not None and 'step_dir' in self.state[p]:
                    p.add_(self.state[p]['step_dir'], alpha=-lr)

        return loss







class ASGDAdam(Optimizer):
    """Proposed ASGD (Improved Adam) with decoupled weight decay."""

    def __init__(self, params, lr=None, beta1=0.9, beta2=0.9, eps=1e-8,
                 lr_min=1e-4, lr_max=3e-4, weight_decay=0):
        defaults = dict(beta1=beta1, beta2=beta2, eps=eps,
                        lr_min=lr_min, lr_max=lr_max,
                        weight_decay=weight_decay)
        super().__init__(params, defaults)
        self.f_min = 0
        self.f_max = 0
        self.last_lr = lr_max

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        total_nonzero_fmin, total_nonzero_fmax = 0, 0

        for group in self.param_groups:
            beta1, beta2, eps = group['beta1'], group['beta2'], group['eps']
            wd = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue

                # Decoupled weight decay
                if wd != 0:
                    p.mul_(1 - wd)

                g = p.grad
                state = self.state[p]

                if not state:
                    state['t'] = 0
                    state['n'] = torch.zeros_like(p)
                    state['H'] = torch.zeros_like(p)

                n, H = state['n'], state['H']
                state['t'] += 1

                n.mul_(beta1).add_(g, alpha=1 - beta1)
                H.mul_(beta2).addcmul_(g, g, value=1 - beta2)

                mean_abs_H = n.abs().mean()
                mean_H = n.mean()

                f_min = (mean_abs_H).sign() - (mean_H).sign()
                f_max = (mean_H).sign() - (mean_abs_H).sign()
                #f_min = (mean_abs_H - mean_H)
               # f_max = (mean_H - mean_abs_H)

                #total_nonzero_fmin += (f_min != 0).sum().item()
                #total_nonzero_fmax += (f_max != 0).sum().item()


                denom = H.sqrt().add(eps)
                state['step_dir'] = n / denom

        #use_lr_min = (total_nonzero_fmax <   total_nonzero_fmin)
        use_lr_min = f_max ==  f_min
        self.f_min = f_min
        self.f_max = f_max

        for group in self.param_groups:
            lr = group['lr_min'] if use_lr_min else group['lr_max']
            self.last_lr = lr

            for p in group['params']:
                if p.grad is not None and 'step_dir' in self.state[p]:
                    p.add_(self.state[p]['step_dir'], alpha=-lr)

        return loss



class Padam(Optimizer):
    """Padam optimizer"""
    def __init__(self, params, lr=1e-3, betas=(0.9,0.999), eps=1e-8, weight_decay=0, amsgrad=False, p=0.125):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, amsgrad=amsgrad, p=p)
        super().__init__(params, defaults)
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad.data
                if grad.is_sparse:
                    raise RuntimeError('Padam does not support sparse gradients')
                state = self.state[p]
                amsgrad = group['amsgrad']
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p.data)
                    state['exp_avg_sq'] = torch.zeros_like(p.data)
                    if amsgrad:
                        state['max_exp_avg_sq'] = torch.zeros_like(p.data)
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                if amsgrad:
                    max_exp_avg_sq = state['max_exp_avg_sq']
                beta1, beta2 = group['betas']
                state['step'] += 1
                exp_avg.mul_(beta1).add_(grad, alpha=1-beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1-beta2)
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                if amsgrad:
                    torch.maximum(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                    denom = (max_exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group['eps'])
                else:
                    denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group['eps'])
                step_size = group['lr'] / bias_correction1
                denom = denom.pow(group['p'])
                if group['weight_decay'] != 0:
                    grad = grad.add(p.data, alpha=group['weight_decay'])
                p.data.addcdiv_(exp_avg, denom, value=-step_size)
        return loss

In [ ]:
# ============================================================
# Unified CTRL(control) + PTPN2(perturbed) CRISPR RL
# MULTI-GENE CRISPRi SCREENING VERSION
# WITH MULTI-STEP WORLD MODEL (PREDICTION HORIZON)
#
# Author: Boabang Francis et al.
# ============================================================

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import sparse
from scipy.spatial.distance import cdist
import math
import matplotlib.pyplot as plt

import scanpy as sc
import anndata as ad

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn

# ============================================================
# GLOBALS
# ============================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ============================================================
# PATHS / HYPERPARAMETERS
# ============================================================

CTRL_MATRIX = "Control_train Data/matrix.mtx"
CTRL_FEAT   = "Control_train Data/genes.tsv"

PTPN2_MATRIX = "Control_train Data/matrix.mtx"
PTPN2_FEAT   = "Control_train Data/genes.tsv"

N_TOP_GENES = 200

GENE_EMB_DIM = 50
N_PCS        = 10
GAT_OUT_DIM  = 32
AE_EPOCHS    = 100
BATCH_SIZE = 256
# ---------- WORLD MODEL ----------
WORLD_HIDDEN = 128
WORLD_EPOCHS = 200
WORLD_LR     = 1e-3
K_NEXT       = 5

#  PREDICTION HORIZON (NEW)
PRED_HORIZON = 2 # number of pseudotime steps ahead

# ============================================================
# 1) LOAD MTX + ALIGN GENES
# ============================================================

def load_mtx(mtx_path, feat_path):
    X = sio.mmread(mtx_path)
    if not sparse.isspmatrix(X):
        X = sparse.coo_matrix(X)
    X = X.T.tocsr()
    genes = pd.read_csv(feat_path, sep="\t", header=None).iloc[:, 0].astype(str).values
    return X, genes

ctrl_X_raw, ctrl_genes = load_mtx(CTRL_MATRIX, CTRL_FEAT)
pt_X_raw, pt_genes     = load_mtx(PTPN2_MATRIX, PTPN2_FEAT)

genes_common, idx_ctrl, idx_pt = np.intersect1d(ctrl_genes, pt_genes, return_indices=True)
ctrl_X_raw = ctrl_X_raw[:, idx_ctrl]
pt_X_raw   = pt_X_raw[:, idx_pt]

X_ctrl_all = ctrl_X_raw.toarray().astype(np.float32)
X_pt_all   = pt_X_raw.toarray().astype(np.float32)

# ============================================================
# 2) TRUE 80/20 PTPN2 SPLIT
# ============================================================

X_pt_tr_raw, X_pt_te_raw = train_test_split(
    X_pt_all, test_size=0.2, random_state=SEED, shuffle=True
)

# ============================================================
# 3) HVG ON PTPN2 TRAIN ONLY
# ============================================================

adata_pt_tr = ad.AnnData(X_pt_tr_raw)
adata_pt_tr.var_names = genes_common

sc.pp.normalize_total(adata_pt_tr, target_sum=1e4)
sc.pp.log1p(adata_pt_tr)
sc.pp.highly_variable_genes(adata_pt_tr, n_top_genes=N_TOP_GENES, subset=True)
sc.pp.scale(adata_pt_tr, max_value=10)

hvg_genes = adata_pt_tr.var_names.values
hvg_mask  = np.isin(genes_common, hvg_genes)

def preprocess_to_hvg(X):
    adata = ad.AnnData(X[:, hvg_mask])
    adata.var_names = hvg_genes
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.scale(adata, max_value=10)
    return adata.X.astype(np.float32)

X_pt_tr = adata_pt_tr.X.astype(np.float32)
X_pt_te = preprocess_to_hvg(X_pt_te_raw)
X_ctrl  = preprocess_to_hvg(X_ctrl_all)

# ============================================================
# 4) GENE AUTOENCODER (PTPN2 TRAIN ONLY)
# ============================================================

class GeneAutoencoder(nn.Module):
    def __init__(self, in_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, in_dim)
        )
    def encode(self, x): return self.encoder(x)
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

ae = GeneAutoencoder(X_pt_tr.shape[1], GENE_EMB_DIM).to(DEVICE)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

Xt = torch.tensor(X_pt_tr, device=DEVICE)
ae.train()
for _ in range(AE_EPOCHS):
    opt.zero_grad()
    recon, _ = ae(Xt)
    loss = loss_fn(recon, Xt)
    loss.backward()
    opt.step()

ae.eval()
with torch.no_grad():
    Z_pt_tr = ae.encode(torch.tensor(X_pt_tr, device=DEVICE)).cpu().numpy()
    Z_pt_te = ae.encode(torch.tensor(X_pt_te, device=DEVICE)).cpu().numpy()
    Z_ctrl  = ae.encode(torch.tensor(X_ctrl,  device=DEVICE)).cpu().numpy()

# ============================================================
# 5) PCA
# ============================================================

pca = PCA(n_components=N_PCS, random_state=SEED)
P_pt_tr = pca.fit_transform(Z_pt_tr)
P_pt_te = pca.transform(Z_pt_te)
P_ctrl  = pca.transform(Z_ctrl)

# ============================================================
# 6) FUSED EMBEDDINGS
# ============================================================

emb_pt_tr = np.concatenate([Z_pt_tr, P_pt_tr], axis=1)
emb_pt_te = np.concatenate([Z_pt_te, P_pt_te], axis=1)
emb_ctrl  = np.concatenate([Z_ctrl,  P_ctrl],  axis=1)

# ============================================================
# 7) PSEUDOTIME (PTPN2 TRAIN)
# ============================================================

sc.pp.neighbors(adata_pt_tr, n_neighbors=15)
sc.tl.diffmap(adata_pt_tr)
adata_pt_tr.uns["iroot"] = len(adata_pt_tr) // 2
sc.tl.dpt(adata_pt_tr)
pt_pt_tr = adata_pt_tr.obs["dpt_pseudotime"].values

def map_to_pt(ref_emb, ref_pt, X):
    nn = cdist(X, ref_emb).argmin(axis=1)
    return ref_pt[nn]

pt_pt_te = map_to_pt(emb_pt_tr, pt_pt_tr, emb_pt_te)

# ============================================================
# 8) MULTI-STEP WORLD PAIRS (🔥 HORIZON)
# ============================================================

def build_world_pairs_horizon(emb, pt, horizon, k=5, min_delta=1e-4):
    Xc, Xn = [], []
    for i in range(len(emb)):
        future = np.where(pt > pt[i] + horizon * min_delta)[0]
        if len(future) < k:
            continue
        dists = np.linalg.norm(emb[future] - emb[i], axis=1)
        nn = future[np.argsort(dists)[:k]]
        Xc.append(emb[i])
        Xn.append(emb[nn].mean(axis=0))
    return np.vstack(Xc), np.vstack(Xn)

Xc_wm, Xn_wm = build_world_pairs_horizon(
    emb_pt_tr, pt_pt_tr, horizon=PRED_HORIZON, k=K_NEXT
)

Xc_te, Xn_te = build_world_pairs_horizon(
    emb_pt_te, pt_pt_te, horizon=PRED_HORIZON, k=K_NEXT
)

# ============================================================
# 9) WORLD MODEL
# ============================================================

class WorldModel(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, dim)
        )
    def forward(self, x): return self.net(x)

world_model = WorldModel(emb_pt_tr.shape[1], WORLD_HIDDEN).to(DEVICE)
opt = torch.optim.Adam(world_model.parameters(), lr=WORLD_LR)
loss_fn = nn.MSELoss()

Xc_t = torch.tensor(Xc_wm, device=DEVICE)
Xn_t = torch.tensor(Xn_wm, device=DEVICE)

world_model.train()
for _ in range(WORLD_EPOCHS):
    opt.zero_grad()
    loss = loss_fn(world_model(Xc_t), Xn_t)
    loss.backward()
    opt.step()

# ============================================================
# 10) WORLD MODEL METRICS
# ============================================================

def metrics(y_true, y_pred):
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Pearson": np.corrcoef(y_true, y_pred)[0,1]
    }

with torch.no_grad():
    pred_tr = world_model(Xc_t).cpu().numpy()
    pred_te = world_model(torch.tensor(Xc_te, device=DEVICE)).cpu().numpy()

rows = []
for split, Yt, Yp in [
    ("Train", Xn_wm, pred_tr),
    ("Test",  Xn_te, pred_te),
]:
    for d in range(Yt.shape[1]):
        m = metrics(Yt[:, d], Yp[:, d])
        m.update({"Split": split, "Dim": d, "Horizon": PRED_HORIZON})
        rows.append(m)

df = pd.DataFrame(rows)
df.to_csv("ptpn2_world_model_metrics_horizon.csv", index=False)

print("\n=== WORLD MODEL METRICS (H =", PRED_HORIZON, ") ===")
print(df.groupby("Split")[["MSE","RMSE","MAE","R2","Pearson"]].mean())



# ============================================================
# 10c) COMPARE OPTIMIZERS FOR PTPN2 WORLD MODEL (HORIZON-AWARE)
# ============================================================

import time
import pandas as pd
import numpy as np
import torch
import math
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---- tensors (already built upstream with horizon-aware pairs) ----
Xc_te_t = torch.tensor(Xc_te, dtype=torch.float32, device=DEVICE)
Xn_te_t = torch.tensor(Xn_te, dtype=torch.float32, device=DEVICE)

# ============================================================
# Metrics
# ============================================================

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    pearson = 0.0 if np.std(y_true) == 0 else float(np.corrcoef(y_true, y_pred)[0, 1])
    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Pearson": pearson,
    }

@torch.no_grad()
def eval_world_model(model, Xc_tr, Xn_tr, Xc_te, Xn_te):
    model.eval()

    pred_tr = model(Xc_tr).cpu().numpy()
    pred_te = model(Xc_te).cpu().numpy()

    rows = []
    for split, Yt, Yp in [
        ("Train", Xn_tr.cpu().numpy(), pred_tr),
        ("Test",  Xn_te.cpu().numpy(), pred_te),
    ]:
        for d in range(Yt.shape[1]):
            m = regression_metrics(Yt[:, d], Yp[:, d])
            m.update({
                "Split": split,
                "Dim": d,
                "Horizon": PRED_HORIZON,
            })
            rows.append(m)

    df_per_dim = pd.DataFrame(rows)
    df_mean = (
        df_per_dim
        .groupby(["Split","Horizon"])[["MSE","RMSE","MAE","R2","Pearson"]]
        .mean()
        .reset_index()
    )
    return df_per_dim, df_mean

# ============================================================
# Optimizer factory
# ============================================================

def make_optimizer(name, params, base_lr):
    n = name.lower()
    if n == "adam":
        return torch.optim.Adam(params, lr=1e-4, betas=(0.9, 0.99))
    if n == "adamw":
        return torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)
    if n == "sgd":
        return torch.optim.SGD(params, lr=1e-2, momentum=0.9, nesterov=True)
    if n == "rmsprop":
        return torch.optim.RMSprop(params, lr=1e-4)

    if n == "asgdadam":
        return ASGDAdam(params, lr_min=5e-5, lr_max=1e-4)
    if n == "asgdamsgrad":
        return ASGDAMSGrad(params, lr_min=5e-5, lr_max=1e-4)
    if n == "padam":
        return Padam(params, lr=1e-2, p=0.125)

    raise ValueError(f"Unknown optimizer: {name}")

# ============================================================
# H-step rollout loss (CRITICAL for horizon training)
# ============================================================

def rollout_loss(model, x0, xH_target, H):
    x = x0
    for _ in range(H):
        x = model(x)
    return nn.functional.mse_loss(x, xH_target)

# ============================================================
# Train one optimizer
# ============================================================

def train_one_optimizer(
    opt_name,
    n_epochs=WORLD_EPOCHS,
    batch_size=BATCH_SIZE,
    base_lr=WORLD_LR,
    seed=SEED,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = WorldModel(emb_pt_tr.shape[1], WORLD_HIDDEN).to(DEVICE)
    optimizer = make_optimizer(opt_name, model.parameters(), base_lr)

    n = Xc_t.shape[0]
    t0 = time.time()
    losses = []

    model.train()
    for epoch in range(n_epochs):
        perm = torch.randperm(n, device=DEVICE)
        total = 0.0

        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            bC = Xc_t[idx]
            bN = Xn_t[idx]

            optimizer.zero_grad()
            loss = rollout_loss(
                model,
                bC,
                bN,
                H=PRED_HORIZON,   # 🔥 horizon enforced here
            )
            loss.backward()
            optimizer.step()

            total += loss.item() * bC.size(0)

        losses.append(total / n)

    train_time = time.time() - t0

    df_dim, df_mean = eval_world_model(
        model, Xc_t, Xn_t, Xc_te_t, Xn_te_t
    )

    df_mean["Optimizer"] = opt_name
    df_mean["TrainTimeSec"] = train_time
    df_mean["FinalTrainLoss"] = losses[-1]

    df_dim["Optimizer"] = opt_name
    df_dim["TrainTimeSec"] = train_time

    last_lr = getattr(optimizer, "last_lr", None)
    if last_lr is not None:
        df_mean["LastLR"] = float(last_lr)
        df_dim["LastLR"] = float(last_lr)

    return df_dim, df_mean

# ============================================================
# Run comparison
# ============================================================

optimizers_to_try = [
    "Adam",
   "AdamW",
    "RMSprop",
    "SGD",
    "Padam",
    "ASGDAdam",
    "ASGDAMSGrad",
]

all_dim, all_mean = [], []

for opt_name in optimizers_to_try:
    print(f"\n[OPT | H={PRED_HORIZON}] Training with {opt_name}")
    d_dim, d_mean = train_one_optimizer(opt_name)
    all_dim.append(d_dim)
    all_mean.append(d_mean)

df_compare_per_dim = pd.concat(all_dim, ignore_index=True)
df_compare_mean = pd.concat(all_mean, ignore_index=True)

# rank by TEST RMSE
ranked = (
    df_compare_mean[df_compare_mean["Split"] == "Test"]
    .sort_values(["RMSE","MSE"])
)

print("\n=== OPTIMIZER COMPARISON (H =", PRED_HORIZON, ") ===")
print(ranked.to_string(index=False))

df_compare_per_dim.to_csv(
    f"ptpn2_world_optimizer_compare_per_dim_H{PRED_HORIZON}.csv",
    index=False
)
df_compare_mean.to_csv(
    f"ptpn2_world_optimizer_compare_mean_H{PRED_HORIZON}.csv",
    index=False
)

print("\nSaved horizon-aware results.")










Using device: cuda


/tmp/ipykernel_6951/3309576696.py:180: UserWarning: You’re trying to run this on 200 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  sc.pp.neighbors(adata_pt_tr, n_neighbors=15)



=== WORLD MODEL METRICS (H = 2 ) ===
            MSE      RMSE       MAE        R2   Pearson
Split                                                  
Test   0.081699  0.278300  0.211253  0.691310  0.842631
Train  0.063725  0.248544  0.191189  0.828972  0.909577

[OPT | H=2] Training with Adam

[OPT | H=2] Training with AdamW

[OPT | H=2] Training with RMSprop

[OPT | H=2] Training with SGD

[OPT | H=2] Training with Padam

[OPT | H=2] Training with ASGDAdam

[OPT | H=2] Training with ASGDAMSGrad

=== OPTIMIZER COMPARISON (H = 2 ) ===
Split  Horizon      MSE     RMSE      MAE        R2  Pearson   Optimizer  TrainTimeSec  FinalTrainLoss  LastLR
 Test        2 0.794754 0.759339 0.604572 -1.147521 0.150654       Padam      3.882504        0.101670     NaN
 Test        2 1.004950 0.864504 0.692533 -1.795629 0.105553         SGD      2.781886        0.063756     NaN
 Test        2 1.426534 1.026895 0.847267 -3.043937 0.066815 ASGDAMSGrad      5.127492        0.073814 0.00005
 Test        2 1

In [ ]:
### New code

In [ ]:
import os
import zipfile
from google.colab import files

zip_name = "cyokines_all_files.zip"
root_dir = "/content"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for foldername, subfolders, filenames in os.walk(root_dir):
        for filename in filenames:
            file_path = os.path.join(foldername, filename)
            arcname = os.path.relpath(file_path, root_dir)
            zipf.write(file_path, arcname)

files.download(zip_name)




OSError: [Errno 28] No space left on device

In [ ]:
# ============================================================
# Unified CTRL(control) + PTPN2(perturbed) CRISPR RL
# MULTI-GENE CRISPRi SCREENING VERSION
# WITH MULTI-STEP WORLD MODEL (PREDICTION HORIZON)
#
# Author: Boabang Francis et al.
# ============================================================

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import sparse
from scipy.spatial.distance import cdist
import math
import matplotlib.pyplot as plt
import time

import scanpy as sc
import anndata as ad

from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn

# ============================================================
# GLOBALS
# ============================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ============================================================
# PATHS / HYPERPARAMETERS
# ============================================================

CTRL_MATRIX = "Control_train Data/matrix.mtx"
CTRL_FEAT   = "Control_train Data/genes.tsv"

PTPN2_MATRIX = "Perturbed_train Data/matrix.mtx"
PTPN2_FEAT   = "Perturbed_train Data/genes.tsv"

# requested base values
N_TOP_GENES = 2000
GENE_EMB_DIM = 500

N_PCS      = 10
GAT_OUT_DIM  = 32
AE_EPOCHS    = 500
BATCH_SIZE   = 256

# ---------- WORLD MODEL ----------
WORLD_HIDDEN = 1024
WORLD_EPOCHS = 500
WORLD_LR     = 1e-5
K_NEXT       = 5

# ---------- PREDICTION HORIZON ----------
PRED_HORIZON = 20

# ---------- FOLDS ----------
N_FOLDS = 5

HORIZON_LIST = [1, 3,  5, 10, 20]

# ---------- SDI WEIGHTS ----------
W1, W2, W3 = 0.3, 0.3, 0.4

# ============================================================
# 1) LOAD MTX + ALIGN GENES
# ============================================================

def load_mtx(mtx_path, feat_path):
    X = sio.mmread(mtx_path)
    if not sparse.isspmatrix(X):
        X = sparse.coo_matrix(X)
    X = X.T.tocsr()
    genes = pd.read_csv(feat_path, sep="\t", header=None).iloc[:, 0].astype(str).values
    return X, genes

ctrl_X_raw, ctrl_genes = load_mtx(CTRL_MATRIX, CTRL_FEAT)
pt_X_raw, pt_genes     = load_mtx(PTPN2_MATRIX, PTPN2_FEAT)

genes_common, idx_ctrl, idx_pt = np.intersect1d(ctrl_genes, pt_genes, return_indices=True)
ctrl_X_raw = ctrl_X_raw[:, idx_ctrl]
pt_X_raw   = pt_X_raw[:, idx_pt]

X_ctrl_all = ctrl_X_raw.toarray().astype(np.float32)
X_pt_all   = pt_X_raw.toarray().astype(np.float32)

TOTAL_NUM_GENES = X_pt_all.shape[1]

# ============================================================
# 2) K-FOLD SPLIT SETUP
# ============================================================

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ============================================================
# 3) MULTI-STEP WORLD PAIRS (HORIZON)
# ============================================================

def build_world_pairs_horizon(emb, pt, horizon, k=5, min_delta=1e-4):
    Xc, Xn = [], []
    for i in range(len(emb)):
        future = np.where(pt > pt[i] + horizon * min_delta)[0]
        if len(future) < k:
            continue
        dists = np.linalg.norm(emb[future] - emb[i], axis=1)
        nn_idx = future[np.argsort(dists)[:k]]
        Xc.append(emb[i])
        Xn.append(emb[nn_idx].mean(axis=0))

    if len(Xc) == 0 or len(Xn) == 0:
        return np.empty((0, emb.shape[1]), dtype=np.float32), np.empty((0, emb.shape[1]), dtype=np.float32)

    return np.vstack(Xc).astype(np.float32), np.vstack(Xn).astype(np.float32)

# ============================================================
# 4) WORLD MODEL
# ============================================================

class WorldModel(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, dim)
        )

    def forward(self, x):
        return self.net(x)

# ============================================================
# 5) SDI / VARIANCE / STD HELPERS
# ============================================================

def compute_distribution_stats(arr):
    arr = np.asarray(arr).ravel()
    return {
        "Variance": float(np.var(arr)),
        "StdDev": float(np.std(arr))
    }

def compute_h_cell_from_embedding(Z, n_bins=10):
    x = Z[:, 0]
    counts, _ = np.histogram(x, bins=n_bins)
    counts = counts.astype(np.float64)
    counts = counts[counts > 0]
    if len(counts) == 0:
        return 0.0
    pi = counts / counts.sum()
    H_cell = -np.sum(pi * np.log(pi + 1e-12))
    H_cell = H_cell / np.log(n_bins + 1e-12)
    return float(H_cell)

def compute_cdi_from_embedding(Z):
    H_vec = np.mean(Z ** 2, axis=0)
    mu_H = np.mean(H_vec)
    sigma_H = np.std(H_vec)
    return float(sigma_H / (mu_H + 1e-8))

def compute_d_latent(Z):
    z_bar = np.mean(Z, axis=0)
    return float(np.mean(np.linalg.norm(Z - z_bar, axis=1)))

def compute_sdi_components(Z):
    d_latent = compute_d_latent(Z)
    h_cell   = compute_h_cell_from_embedding(Z)
    cdi      = compute_cdi_from_embedding(Z)

    flat_stats = compute_distribution_stats(Z)
    return {
        "D_latent": d_latent,
        "H_cell": h_cell,
        "CDI": cdi,
        "Variance": flat_stats["Variance"],
        "StdDev": flat_stats["StdDev"],
    }

def compute_reference_stats(Z_train_ref, Z_test_ref):
    train_ref = compute_sdi_components(Z_train_ref)
    test_ref  = compute_sdi_components(Z_test_ref)

    return {
        "D_latent_ref": max(train_ref["D_latent"], test_ref["D_latent"], 1e-12),
        "CDI_ref": max(train_ref["CDI"], test_ref["CDI"], 1e-12),
        "Variance_ref": max(train_ref["Variance"], test_ref["Variance"], 1e-12),
        "StdDev_ref": max(train_ref["StdDev"], test_ref["StdDev"], 1e-12),
    }

def compute_sdi_value(
    D_latent,
    H_cell,
    CDI,
    horizon,
    D_latent_ref,
    CDI_ref,
    variance=None,
    stddev=None,
    variance_ref=None,
    stddev_ref=None,
    w1=W1,
    w2=W2,
    w3=W3,
    use_variance_boost=True,
    lambda_var=0.05,
    lambda_std=0.05
):
    D_norm   = D_latent / (D_latent_ref + 1e-12)
    CDI_norm = CDI / (CDI_ref + 1e-12)

    base_sdi = w1 * D_norm + w2 * H_cell + w3 * CDI_norm
    horizon_scale = np.sqrt(max(horizon, 1))
    sdi_scaled = base_sdi * horizon_scale

    if use_variance_boost and variance is not None and stddev is not None:
        var_norm = variance / (variance_ref + 1e-12)
        std_norm = stddev / (stddev_ref + 1e-12)
        sdi_scaled = sdi_scaled + lambda_var * var_norm + lambda_std * std_norm

    return float(sdi_scaled)

# ============================================================
# 5b) METRICS
# ============================================================

def metrics(y_true, y_pred):
    pearson = 0.0 if np.std(y_true) == 0 else float(np.corrcoef(y_true, y_pred)[0, 1])

    base = {
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Pearson": pearson
    }

    residual = y_true - y_pred
    base.update(compute_distribution_stats(residual))
    return base

# ============================================================
# 5c) EVALUATION HELPERS
# ============================================================

@torch.no_grad()
def evaluate_model_with_sdi(
    model,
    Xc_tr,
    Xn_tr,
    Xc_te,
    Xn_te,
    horizon,
    ref_stats,
    fold_id,
    fold_n_top_genes,
    fold_gene_emb_dim
):
    model.eval()

    pred_tr = model(Xc_tr).cpu().numpy()
    pred_te = model(Xc_te).cpu().numpy()

    sdi_train = compute_sdi_components(Xn_tr.cpu().numpy())
    sdi_test  = compute_sdi_components(Xn_te.cpu().numpy())

    sdi_train["SDI"] = compute_sdi_value(
        D_latent=sdi_train["D_latent"],
        H_cell=sdi_train["H_cell"],
        CDI=sdi_train["CDI"],
        horizon=horizon,
        D_latent_ref=ref_stats["D_latent_ref"],
        CDI_ref=ref_stats["CDI_ref"],
        variance=sdi_train["Variance"],
        stddev=sdi_train["StdDev"],
        variance_ref=ref_stats["Variance_ref"],
        stddev_ref=ref_stats["StdDev_ref"],
    )

    sdi_test["SDI"] = compute_sdi_value(
        D_latent=sdi_test["D_latent"],
        H_cell=sdi_test["H_cell"],
        CDI=sdi_test["CDI"],
        horizon=horizon,
        D_latent_ref=ref_stats["D_latent_ref"],
        CDI_ref=ref_stats["CDI_ref"],
        variance=sdi_test["Variance"],
        stddev=sdi_test["StdDev"],
        variance_ref=ref_stats["Variance_ref"],
        stddev_ref=ref_stats["StdDev_ref"],
    )

    rows = []
    split_sdi_map = {"Train": sdi_train, "Test": sdi_test}

    for split, Yt, Yp in [
        ("Train", Xn_tr.cpu().numpy(), pred_tr),
        ("Test",  Xn_te.cpu().numpy(), pred_te),
    ]:
        for d in range(Yt.shape[1]):
            m = metrics(Yt[:, d], Yp[:, d])
            m.update({
                "Split": split,
                "Dim": d,
                "Fold": fold_id,
                "Horizon": horizon,
                "Fold_N_TOP_GENES": fold_n_top_genes,
                "Fold_GENE_EMB_DIM": fold_gene_emb_dim,
                "TargetVariance": split_sdi_map[split]["Variance"],
                "TargetStdDev": split_sdi_map[split]["StdDev"],
                "D_latent": split_sdi_map[split]["D_latent"],
                "H_cell": split_sdi_map[split]["H_cell"],
                "CDI": split_sdi_map[split]["CDI"],
                "SDI": split_sdi_map[split]["SDI"],
            })
            rows.append(m)

    return pd.DataFrame(rows)

# ============================================================
# 5d) OPTIMIZER FACTORY
# ============================================================

def make_optimizer(name, params, base_lr):
    n = name.lower()
    if n == "adam":
        return torch.optim.Adam(params, lr=1e-4, betas=(0.9, 0.99))
    if n == "adamw":
        return torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)
    if n == "sgd":
        return torch.optim.SGD(params, lr=1e-3, momentum=0.9, nesterov=True)
    if n == "rmsprop":
        return torch.optim.RMSprop(params, lr=1e-4)

    if n == "asgdadam":
        return ASGDAdam(params, lr_min=5e-5, lr_max=1e-4)
    if n == "asgdamsgrad":
        return ASGDAMSGrad(params, lr_min=5e-5, lr_max=1e-4)
    if n == "padam":
        return Padam(params, lr=1e-3, p=0.125)

    raise ValueError(f"Unknown optimizer: {name}")

def rollout_loss(model, x0, xH_target, H):
    x = x0
    for _ in range(H):
        x = model(x)
    return nn.functional.mse_loss(x, xH_target)

# ============================================================
# 6) GENE AUTOENCODER
# ============================================================

class GeneAutoencoder(nn.Module):
    def __init__(self, in_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, in_dim)
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

# ============================================================
# 7) FOLD PREPARATION PIPELINE
# ============================================================

def prepare_fold_data(X_pt_tr_raw, X_pt_te_raw, fold_id):
    torch.manual_seed(SEED + fold_id)
    np.random.seed(SEED + fold_id)

    # --------------------------------------------------------
    # pick N_TOP_GENES per fold from the full gene space
    # --------------------------------------------------------
    fold_n_top_genes = min(N_TOP_GENES, TOTAL_NUM_GENES)

    # ---------------- HVG on fold-train only ----------------
    adata_pt_tr = ad.AnnData(X_pt_tr_raw)
    adata_pt_tr.var_names = genes_common

    sc.pp.normalize_total(adata_pt_tr, target_sum=1e4)
    sc.pp.log1p(adata_pt_tr)
    sc.pp.highly_variable_genes(
        adata_pt_tr,
        n_top_genes=fold_n_top_genes,
        subset=True
    )
    sc.pp.scale(adata_pt_tr, max_value=10)

    hvg_genes = adata_pt_tr.var_names.values
    hvg_mask  = np.isin(genes_common, hvg_genes)

    # --------------------------------------------------------
    # pick GENE_EMB_DIM per fold after fold-specific HVG selection
    # --------------------------------------------------------
    fold_gene_emb_dim = min(GENE_EMB_DIM, len(hvg_genes))

    def preprocess_to_hvg(X):
        adata = ad.AnnData(X[:, hvg_mask])
        adata.var_names = hvg_genes
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.scale(adata, max_value=10)
        return adata.X.astype(np.float32)

    X_pt_tr = adata_pt_tr.X.astype(np.float32)
    X_pt_te = preprocess_to_hvg(X_pt_te_raw)
    X_ctrl  = preprocess_to_hvg(X_ctrl_all)

    # ---------------- Autoencoder on fold-train only ----------------
    ae = GeneAutoencoder(X_pt_tr.shape[1], fold_gene_emb_dim).to(DEVICE)
    opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    Xt = torch.tensor(X_pt_tr, dtype=torch.float32, device=DEVICE)
    ae.train()
    for _ in range(AE_EPOCHS):
        opt.zero_grad()
        recon, _ = ae(Xt)
        loss = loss_fn(recon, Xt)
        loss.backward()
        opt.step()

    ae.eval()
    with torch.no_grad():
        Z_pt_tr = ae.encode(torch.tensor(X_pt_tr, dtype=torch.float32, device=DEVICE)).cpu().numpy()
        Z_pt_te = ae.encode(torch.tensor(X_pt_te, dtype=torch.float32, device=DEVICE)).cpu().numpy()
        Z_ctrl  = ae.encode(torch.tensor(X_ctrl,  dtype=torch.float32, device=DEVICE)).cpu().numpy()

    # ---------------- PCA fit on fold-train only ----------------
    fold_n_pcs = min(N_PCS, Z_pt_tr.shape[1], Z_pt_tr.shape[0])
    pca = PCA(n_components=fold_n_pcs, random_state=SEED)
    P_pt_tr = pca.fit_transform(Z_pt_tr)
    P_pt_te = pca.transform(Z_pt_te)
    P_ctrl  = pca.transform(Z_ctrl)

    # ---------------- Fused embeddings ----------------
    emb_pt_tr = np.concatenate([Z_pt_tr, P_pt_tr], axis=1).astype(np.float32)
    emb_pt_te = np.concatenate([Z_pt_te, P_pt_te], axis=1).astype(np.float32)
    emb_ctrl  = np.concatenate([Z_ctrl,  P_ctrl],  axis=1).astype(np.float32)

    # ---------------- Pseudotime on fold-train only ----------------
    sc.pp.neighbors(adata_pt_tr, n_neighbors=15)
    sc.tl.diffmap(adata_pt_tr)
    adata_pt_tr.uns["iroot"] = len(adata_pt_tr) // 2
    sc.tl.dpt(adata_pt_tr)
    pt_pt_tr = adata_pt_tr.obs["dpt_pseudotime"].values.astype(np.float32)

    def map_to_pt(ref_emb, ref_pt, X):
        nn_idx = cdist(X, ref_emb).argmin(axis=1)
        return ref_pt[nn_idx].astype(np.float32)

    pt_pt_te = map_to_pt(emb_pt_tr, pt_pt_tr, emb_pt_te)

    # ---------------- Reference stats from base horizon ----------------
    base_horizon = min(HORIZON_LIST)

    Xc_ref, Xn_ref = build_world_pairs_horizon(
        emb_pt_tr, pt_pt_tr, horizon=base_horizon, k=K_NEXT
    )
    Xc_ref_te, Xn_ref_te = build_world_pairs_horizon(
        emb_pt_te, pt_pt_te, horizon=base_horizon, k=K_NEXT
    )

    if len(Xn_ref) == 0 or len(Xn_ref_te) == 0:
        ref_stats = {
            "D_latent_ref": 1e-12,
            "CDI_ref": 1e-12,
            "Variance_ref": 1e-12,
            "StdDev_ref": 1e-12,
        }
    else:
        ref_stats = compute_reference_stats(Xn_ref, Xn_ref_te)

    return {
        "emb_pt_tr": emb_pt_tr,
        "emb_pt_te": emb_pt_te,
        "emb_ctrl": emb_ctrl,
        "pt_pt_tr": pt_pt_tr,
        "pt_pt_te": pt_pt_te,
        "ref_stats": ref_stats,
        "input_dim": emb_pt_tr.shape[1],
        "fold_n_top_genes": fold_n_top_genes,
        "fold_gene_emb_dim": fold_gene_emb_dim,
        "fold_n_pcs": fold_n_pcs,
        "hvg_genes": hvg_genes,
    }

# ============================================================
# 8) TRAIN ONE OPTIMIZER / ONE FOLD / ONE HORIZON
# ============================================================

def train_one_optimizer_for_horizon(
    opt_name,
    horizon,
    Xc_wm_h,
    Xn_wm_h,
    Xc_te_h,
    Xn_te_h,
    ref_stats,
    fold_id,
    input_dim,
    fold_n_top_genes,
    fold_gene_emb_dim,
    n_epochs=WORLD_EPOCHS,
    batch_size=BATCH_SIZE,
    base_lr=WORLD_LR,
    seed=SEED,
):
    torch.manual_seed(seed + fold_id + horizon)
    np.random.seed(seed + fold_id + horizon)

    model = WorldModel(input_dim, WORLD_HIDDEN).to(DEVICE)
    optimizer = make_optimizer(opt_name, model.parameters(), base_lr)

    Xc_t_h    = torch.tensor(Xc_wm_h, dtype=torch.float32, device=DEVICE)
    Xn_t_h    = torch.tensor(Xn_wm_h, dtype=torch.float32, device=DEVICE)
    Xc_te_t_h = torch.tensor(Xc_te_h, dtype=torch.float32, device=DEVICE)
    Xn_te_t_h = torch.tensor(Xn_te_h, dtype=torch.float32, device=DEVICE)

    n = Xc_t_h.shape[0]
    losses = []
    t0 = time.time()

    model.train()
    for epoch in range(n_epochs):
        perm = torch.randperm(n, device=DEVICE)
        total = 0.0

        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            bC = Xc_t_h[idx]
            bN = Xn_t_h[idx]

            optimizer.zero_grad()
            loss = rollout_loss(model, bC, bN, H=horizon)
            loss.backward()
            optimizer.step()

            total += loss.item() * bC.size(0)

        losses.append(total / n)

    train_time = time.time() - t0

    df_dim = evaluate_model_with_sdi(
        model=model,
        Xc_tr=Xc_t_h,
        Xn_tr=Xn_t_h,
        Xc_te=Xc_te_t_h,
        Xn_te=Xn_te_t_h,
        horizon=horizon,
        ref_stats=ref_stats,
        fold_id=fold_id,
        fold_n_top_genes=fold_n_top_genes,
        fold_gene_emb_dim=fold_gene_emb_dim,
    )

    df_dim["Optimizer"] = opt_name
    df_dim["TrainTimeSec"] = train_time
    df_dim["FinalTrainLoss"] = losses[-1]
    df_dim["Fold"] = fold_id
    df_dim["Fold_N_TOP_GENES"] = fold_n_top_genes
    df_dim["Fold_GENE_EMB_DIM"] = fold_gene_emb_dim

    last_lr = getattr(optimizer, "last_lr", None)
    if last_lr is not None:
        df_dim["LastLR"] = float(last_lr)

    return df_dim

# ============================================================
# 9) RUN ALL HORIZONS × FOLDS × OPTIMIZERS
# ============================================================

optimizers_to_try = [
    "AdamW",
    "RMSprop",
    "SGD",
    "Padam",
    "ASGDAdam",
    "Adam",
    "ASGDAMSGrad",
]

all_rows = []
fold_config_rows = []

for fold_id, (tr_idx, te_idx) in enumerate(kf.split(X_pt_all), start=1):
    print(f"\n================ FOLD = {fold_id}/{N_FOLDS} ================")

    X_pt_tr_raw = X_pt_all[tr_idx]
    X_pt_te_raw = X_pt_all[te_idx]

    fold_data = prepare_fold_data(X_pt_tr_raw, X_pt_te_raw, fold_id=fold_id)

    emb_pt_tr = fold_data["emb_pt_tr"]
    emb_pt_te = fold_data["emb_pt_te"]
    pt_pt_tr  = fold_data["pt_pt_tr"]
    pt_pt_te  = fold_data["pt_pt_te"]
    ref_stats = fold_data["ref_stats"]
    input_dim = fold_data["input_dim"]
    fold_n_top_genes = fold_data["fold_n_top_genes"]
    fold_gene_emb_dim = fold_data["fold_gene_emb_dim"]
    fold_n_pcs = fold_data["fold_n_pcs"]

    fold_config_rows.append({
        "Fold": fold_id,
        "Fold_N_TOP_GENES": fold_n_top_genes,
        "Fold_GENE_EMB_DIM": fold_gene_emb_dim,
        "Fold_N_PCS": fold_n_pcs,
        "TrainCells": len(tr_idx),
        "TestCells": len(te_idx),
        "TotalGenes": TOTAL_NUM_GENES
    })

    print(
        f"Fold {fold_id}: "
        f"N_TOP_GENES={fold_n_top_genes}, "
        f"GENE_EMB_DIM={fold_gene_emb_dim}, "
        f"N_PCS={fold_n_pcs}"
    )

    for horizon in HORIZON_LIST:
        print(f"\n[FOLD {fold_id}/{N_FOLDS} | H={horizon}]")

        Xc_wm_h, Xn_wm_h = build_world_pairs_horizon(
            emb_pt_tr, pt_pt_tr, horizon=horizon, k=K_NEXT
        )
        Xc_te_h, Xn_te_h = build_world_pairs_horizon(
            emb_pt_te, pt_pt_te, horizon=horizon, k=K_NEXT
        )

        if len(Xc_wm_h) == 0 or len(Xc_te_h) == 0:
            print(f"Skipping fold={fold_id}, horizon={horizon} because no valid pairs were found.")
            continue

        for opt_name in optimizers_to_try:
            print(f"[OPT | FOLD={fold_id} | H={horizon}] Training with {opt_name}")

            df_dim = train_one_optimizer_for_horizon(
                opt_name=opt_name,
                horizon=horizon,
                Xc_wm_h=Xc_wm_h,
                Xn_wm_h=Xn_wm_h,
                Xc_te_h=Xc_te_h,
                Xn_te_h=Xn_te_h,
                ref_stats=ref_stats,
                fold_id=fold_id,
                input_dim=input_dim,
                fold_n_top_genes=fold_n_top_genes,
                fold_gene_emb_dim=fold_gene_emb_dim,
            )
            all_rows.append(df_dim)

if len(all_rows) == 0:
    raise ValueError("No valid fold/horizon pairs were generated. Check pseudotime and horizon settings.")

df_compare_per_dim = pd.concat(all_rows, ignore_index=True)
df_fold_config = pd.DataFrame(fold_config_rows)

# ============================================================
# 10) AVERAGE ACROSS FOLDS
# ============================================================

df_compare_mean = (
    df_compare_per_dim
    .groupby(["Optimizer", "Split", "Horizon"], as_index=False)[[
        "MSE", "RMSE", "MAE", "R2", "Pearson",
        "Variance", "StdDev",
        "TargetVariance", "TargetStdDev",
        "D_latent", "H_cell", "CDI", "SDI",
        "TrainTimeSec", "FinalTrainLoss",
        "Fold_N_TOP_GENES", "Fold_GENE_EMB_DIM"
    ]]
    .mean()
)

df_compare_std = (
    df_compare_per_dim
    .groupby(["Optimizer", "Split", "Horizon"], as_index=False)[[
        "MSE", "RMSE", "MAE", "R2", "Pearson",
        "Variance", "StdDev",
        "TargetVariance", "TargetStdDev",
        "D_latent", "H_cell", "CDI", "SDI",
        "Fold_N_TOP_GENES", "Fold_GENE_EMB_DIM"
    ]]
    .std()
    .rename(columns={
        "MSE": "MSE_std",
        "RMSE": "RMSE_std",
        "MAE": "MAE_std",
        "R2": "R2_std",
        "Pearson": "Pearson_std",
        "Variance": "Variance_std",
        "StdDev": "StdDev_std",
        "TargetVariance": "TargetVariance_std",
        "TargetStdDev": "TargetStdDev_std",
        "D_latent": "D_latent_std",
        "H_cell": "H_cell_std",
        "CDI": "CDI_std",
        "SDI": "SDI_std",
        "Fold_N_TOP_GENES": "Fold_N_TOP_GENES_std",
        "Fold_GENE_EMB_DIM": "Fold_GENE_EMB_DIM_std",
    })
)

df_compare_mean = df_compare_mean.merge(
    df_compare_std,
    on=["Optimizer", "Split", "Horizon"],
    how="left"
)

ranked = (
    df_compare_mean[df_compare_mean["Split"] == "Test"]
    .sort_values(["Horizon", "RMSE", "MSE"])
)

print("\n=== OPTIMIZER COMPARISON: AVERAGED ACROSS FOLDS ===")
print(ranked.to_string(index=False))

print("\n=== FOLD CONFIGURATION SUMMARY ===")
print(df_fold_config.to_string(index=False))

df_compare_per_dim.to_csv("ptpn2_world_optimizer_compare_per_dim_all_folds.csv", index=False)
df_compare_mean.to_csv("ptpn2_world_optimizer_compare_mean_all_folds.csv", index=False)
df_fold_config.to_csv("ptpn2_fold_configuration_summary.csv", index=False)

# ============================================================
# 11) SIMPLE HORIZON SUMMARY
# ============================================================

horizon_summary = (
    df_compare_mean[df_compare_mean["Split"] == "Test"]
    .groupby(["Horizon"], as_index=False)[[
        "D_latent", "H_cell", "CDI", "SDI",
        "TargetVariance", "TargetStdDev",
        "Fold_N_TOP_GENES", "Fold_GENE_EMB_DIM"
    ]]
    .mean()
)

print("\n=== TEST HORIZON SUMMARY ===")
print(horizon_summary.to_string(index=False))

horizon_summary.to_csv("ptpn2_horizon_sdi_summary.csv", index=False)

print("\nSaved horizon-aware, fold-averaged SDI/variance/std results.")

Using device: cuda

================ FOLD = 1/5 ================


/tmp/ipykernel_6932/2476492068.py:459: UserWarning: You’re trying to run this on 2000 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  sc.pp.neighbors(adata_pt_tr, n_neighbors=15)


Fold 1: N_TOP_GENES=2000, GENE_EMB_DIM=500, N_PCS=10

[FOLD 1/5 | H=1]
[OPT | FOLD=1 | H=1] Training with AdamW
[OPT | FOLD=1 | H=1] Training with RMSprop
[OPT | FOLD=1 | H=1] Training with SGD
[OPT | FOLD=1 | H=1] Training with Padam
[OPT | FOLD=1 | H=1] Training with ASGDAdam
[OPT | FOLD=1 | H=1] Training with Adam
[OPT | FOLD=1 | H=1] Training with ASGDAMSGrad

[FOLD 1/5 | H=3]
[OPT | FOLD=1 | H=3] Training with AdamW
[OPT | FOLD=1 | H=3] Training with RMSprop
[OPT | FOLD=1 | H=3] Training with SGD
[OPT | FOLD=1 | H=3] Training with Padam
[OPT | FOLD=1 | H=3] Training with ASGDAdam
[OPT | FOLD=1 | H=3] Training with Adam
[OPT | FOLD=1 | H=3] Training with ASGDAMSGrad

[FOLD 1/5 | H=5]
[OPT | FOLD=1 | H=5] Training with AdamW
[OPT | FOLD=1 | H=5] Training with RMSprop
[OPT | FOLD=1 | H=5] Training with SGD
[OPT | FOLD=1 | H=5] Training with Padam
[OPT | FOLD=1 | H=5] Training with ASGDAdam
[OPT | FOLD=1 | H=5] Training with Adam
[OPT | FOLD=1 | H=5] Training with ASGDAMSGrad

[FOLD 1

/tmp/ipykernel_6932/2476492068.py:459: UserWarning: You’re trying to run this on 2000 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  sc.pp.neighbors(adata_pt_tr, n_neighbors=15)


Fold 2: N_TOP_GENES=2000, GENE_EMB_DIM=500, N_PCS=10

[FOLD 2/5 | H=1]
[OPT | FOLD=2 | H=1] Training with AdamW
[OPT | FOLD=2 | H=1] Training with RMSprop
[OPT | FOLD=2 | H=1] Training with SGD
[OPT | FOLD=2 | H=1] Training with Padam
[OPT | FOLD=2 | H=1] Training with ASGDAdam
[OPT | FOLD=2 | H=1] Training with Adam
[OPT | FOLD=2 | H=1] Training with ASGDAMSGrad

[FOLD 2/5 | H=3]
[OPT | FOLD=2 | H=3] Training with AdamW
[OPT | FOLD=2 | H=3] Training with RMSprop
[OPT | FOLD=2 | H=3] Training with SGD
[OPT | FOLD=2 | H=3] Training with Padam
[OPT | FOLD=2 | H=3] Training with ASGDAdam
[OPT | FOLD=2 | H=3] Training with Adam
[OPT | FOLD=2 | H=3] Training with ASGDAMSGrad

[FOLD 2/5 | H=5]
[OPT | FOLD=2 | H=5] Training with AdamW
[OPT | FOLD=2 | H=5] Training with RMSprop
[OPT | FOLD=2 | H=5] Training with SGD
[OPT | FOLD=2 | H=5] Training with Padam
[OPT | FOLD=2 | H=5] Training with ASGDAdam
[OPT | FOLD=2 | H=5] Training with Adam
[OPT | FOLD=2 | H=5] Training with ASGDAMSGrad

[FOLD 2

...

Fold 3: N_TOP_GENES=2000, GENE_EMB_DIM=500, N_PCS=10

[FOLD 3/5 | H=1]
[OPT | FOLD=3 | H=1] Training with AdamW
[OPT | FOLD=3 | H=1] Training with RMSprop
[OPT | FOLD=3 | H=1] Training with SGD
[OPT | FOLD=3 | H=1] Training with Padam
[OPT | FOLD=3 | H=1] Training with ASGDAdam
[OPT | FOLD=3 | H=1] Training with Adam
[OPT | FOLD=3 | H=1] Training with ASGDAMSGrad

[FOLD 3/5 | H=3]
[OPT | FOLD=3 | H=3] Training with AdamW
[OPT | FOLD=3 | H=3] Training with RMSprop
[OPT | FOLD=3 | H=3] Training with SGD
[OPT | FOLD=3 | H=3] Training with Padam
[OPT | FOLD=3 | H=3] Training with ASGDAdam
[OPT | FOLD=3 | H=3] Training with Adam
[OPT | FOLD=3 | H=3] Training with ASGDAMSGrad

[FOLD 3/5 | H=5]
[OPT | FOLD=3 | H=5] Training with AdamW
[OPT | FOLD=3 | H=5] Training with RMSprop
[OPT | FOLD=3 | H=5] Training with SGD
[OPT | FOLD=3 | H=5] Training with Padam
[OPT | FOLD=3 | H=5] Training with ASGDAdam
[OPT | FOLD=3 | H=5] Training with Adam
[OPT | FOLD=3 | H=5] Training with ASGDAMSGrad

[FOLD 3

/tmp/ipykernel_6932/2476492068.py:459: UserWarning: You’re trying to run this on 2000 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  sc.pp.neighbors(adata_pt_tr, n_neighbors=15)


Fold 4: N_TOP_GENES=2000, GENE_EMB_DIM=500, N_PCS=10

[FOLD 4/5 | H=1]
[OPT | FOLD=4 | H=1] Training with AdamW
[OPT | FOLD=4 | H=1] Training with RMSprop
[OPT | FOLD=4 | H=1] Training with SGD
[OPT | FOLD=4 | H=1] Training with Padam
[OPT | FOLD=4 | H=1] Training with ASGDAdam
[OPT | FOLD=4 | H=1] Training with Adam
[OPT | FOLD=4 | H=1] Training with ASGDAMSGrad

[FOLD 4/5 | H=3]
[OPT | FOLD=4 | H=3] Training with AdamW
[OPT | FOLD=4 | H=3] Training with RMSprop
[OPT | FOLD=4 | H=3] Training with SGD
[OPT | FOLD=4 | H=3] Training with Padam
[OPT | FOLD=4 | H=3] Training with ASGDAdam
[OPT | FOLD=4 | H=3] Training with Adam
[OPT | FOLD=4 | H=3] Training with ASGDAMSGrad

[FOLD 4/5 | H=5]
[OPT | FOLD=4 | H=5] Training with AdamW
[OPT | FOLD=4 | H=5] Training with RMSprop
[OPT | FOLD=4 | H=5] Training with SGD
[OPT | FOLD=4 | H=5] Training with Padam
[OPT | FOLD=4 | H=5] Training with ASGDAdam
[OPT | FOLD=4 | H=5] Training with Adam
[OPT | FOLD=4 | H=5] Training with ASGDAMSGrad

[FOLD 4

/tmp/ipykernel_6932/2476492068.py:459: UserWarning: You’re trying to run this on 2000 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  sc.pp.neighbors(adata_pt_tr, n_neighbors=15)


Fold 5: N_TOP_GENES=2000, GENE_EMB_DIM=500, N_PCS=10

[FOLD 5/5 | H=1]
[OPT | FOLD=5 | H=1] Training with AdamW
[OPT | FOLD=5 | H=1] Training with RMSprop
[OPT | FOLD=5 | H=1] Training with SGD
[OPT | FOLD=5 | H=1] Training with Padam
[OPT | FOLD=5 | H=1] Training with ASGDAdam
[OPT | FOLD=5 | H=1] Training with Adam
[OPT | FOLD=5 | H=1] Training with ASGDAMSGrad

[FOLD 5/5 | H=3]
[OPT | FOLD=5 | H=3] Training with AdamW
[OPT | FOLD=5 | H=3] Training with RMSprop
[OPT | FOLD=5 | H=3] Training with SGD
[OPT | FOLD=5 | H=3] Training with Padam
[OPT | FOLD=5 | H=3] Training with ASGDAdam
[OPT | FOLD=5 | H=3] Training with Adam
[OPT | FOLD=5 | H=3] Training with ASGDAMSGrad

[FOLD 5/5 | H=5]
[OPT | FOLD=5 | H=5] Training with AdamW
[OPT | FOLD=5 | H=5] Training with RMSprop
[OPT | FOLD=5 | H=5] Training with SGD
[OPT | FOLD=5 | H=5] Training with Padam
[OPT | FOLD=5 | H=5] Training with ASGDAdam
[OPT | FOLD=5 | H=5] Training with Adam
[OPT | FOLD=5 | H=5] Training with ASGDAMSGrad

[FOLD 5

In [ ]:
pip install scanpy